# STAI-X Challenge — Python sample submission

Minimal, runnable baseline: a **one-dimensional linear regression** that predicts `rate_per_10000_ed_visits` from a single covariate (`gtrends_overdose`).

This notebook demonstrates the submission file format end-to-end:

1. Read the training target + covariates and the validation covariates from `/kaggle/input/staix-challenge/`.
2. Fit `lm(y ~ gtrends_overdose)` on training rows in the three scoring categories (`all_drugs`, `all_opioids`, `all_stimulants`).
3. Predict for every row in `sample_submission.csv`.
4. Write `/kaggle/working/submission.csv` with **only** `row_id` and `rate_per_10000_ed_visits`.

Use this as a template — replace the model with whatever you like.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression

DATA = '/kaggle/input/staix-challenge'

train_y = pd.read_csv(f'{DATA}/train/dose_sys_train.csv')
train_x = pd.read_csv(f'{DATA}/train/covariates.csv')
val_x   = pd.read_csv(f'{DATA}/val/covariates.csv')
sub     = pd.read_csv(f'{DATA}/sample_submission.csv')

print('train_y:', train_y.shape, '| train_x:', train_x.shape, '| val_x:', val_x.shape, '| sub:', sub.shape)

In [ ]:
# Keep only the three scoring categories and join the single predictor.
SCORING = ['all_drugs', 'all_opioids', 'all_stimulants']
FEATURE = 'gtrends_overdose'

train = (
    train_y[train_y['overdose_category'].isin(SCORING)]
    .merge(train_x[['period_id', 'jurisdiction', FEATURE]],
           on=['period_id', 'jurisdiction'], how='left')
    .dropna(subset=['rate_per_10000_ed_visits', FEATURE])
)
print('training rows after filter/merge:', len(train))

In [ ]:
# 1-D linear regression: rate ~ gtrends_overdose.
model = LinearRegression().fit(train[[FEATURE]], train['rate_per_10000_ed_visits'])
print(f'fit: rate = {model.intercept_:.3f} + {model.coef_[0]:.3f} * {FEATURE}')

In [ ]:
# Attach the predictor to the submission template via (period_id, jurisdiction).
val_merged = sub.merge(
    val_x[['period_id', 'jurisdiction', FEATURE]],
    on=['period_id', 'jurisdiction'], how='left',
)

# Fill any missing predictor values with the training mean so predictions stay finite.
val_merged[FEATURE] = val_merged[FEATURE].fillna(train[FEATURE].mean())

sub['rate_per_10000_ed_visits'] = model.predict(val_merged[[FEATURE]])

In [ ]:
# Write the submission with EXACTLY the two required columns.
out = sub[['row_id', 'rate_per_10000_ed_visits']]

assert out.shape == (918, 2)
assert out['row_id'].is_unique
assert np.isfinite(out['rate_per_10000_ed_visits']).all()

out.to_csv('/kaggle/working/submission.csv', index=False)
print('wrote /kaggle/working/submission.csv', out.shape)
out.head()